In [114]:
!pip install -q transformers datasets accelerate evaluate scikit-learn pyarrow

In [115]:
import pandas as pd

from datasets import Dataset
from transformers import AutoTokenizer

In [116]:
BIAS_PATH = "/content/drive/MyDrive/AI-News-Perspective-Analyzer/datasets/article-bias/train_clean.parquet"

STANCE_PATH = "/content/drive/MyDrive/AI-News-Perspective-Analyzer/datasets/stance/train_clean.csv"

bias_df = pd.read_parquet(BIAS_PATH)

stance_df = pd.read_csv(
    STANCE_PATH,
    encoding="latin1"
)

In [117]:
print(bias_df.columns)

print()

print(stance_df.columns)

Index(['topic', 'source', 'bias', 'url', 'title', 'date', 'authors', 'content',
       'content_original', 'source_url', 'bias_text', 'ID', 'content_length'],
      dtype='object')

Index(['Body ID', 'articleBody', 'Headline', 'Stance', 'body_length'], dtype='object')


In [118]:
bias_df = bias_df[["content", "bias"]]

bias_df = bias_df.rename(
    columns={
        "content": "text",
        "bias": "label"
    }
)

bias_df.head()

,text,label
0,"This happens for different reasons , but a key...",0
1,LISTEN TO ARTICLE 5:37 SHARE THIS ARTICLE Shar...,1
2,The Pentagon ’ s top general said Tuesday that...,2
3,Story highlights A Russian lawmaker says Russi...,0
4,Iran 's latest crackdown on freedom includes l...,2


In [119]:
stance_map = {
    "agree": 0,
    "discuss": 1,
    "disagree": 2,
    "unrelated": 3
}

stance_df["label"] = stance_df["Stance"].map(stance_map)

In [120]:
stance_df["text"] = (
    "Headline: "
    + stance_df["Headline"].astype(str)
    + " Body: "
    + stance_df["articleBody"].astype(str)
)

In [121]:
stance_df = stance_df[["text", "label"]]

stance_df.head()

,text,label
0,"Headline: Soldier shot, Parliament locked down...",3
1,Headline: Tourist dubbed ÂSpider ManÂ after ...,3
2,Headline: Luke Somers 'killed in failed rescue...,3
3,Headline: BREAKING: Soldier shot at War Memori...,3
4,Headline: Giant 8ft 9in catfish weighing 19 st...,3


In [122]:
print(type(stance_df.loc[0, "text"]))

print(type(bias_df.loc[0, "text"]))

<class 'str'>
<class 'str'>


In [123]:
bias_dataset = Dataset.from_pandas(
    bias_df,
    preserve_index=False
)

stance_dataset = Dataset.from_pandas(
    stance_df,
    preserve_index=False
)

In [124]:
MODEL_NAME = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

In [125]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

In [126]:
tokenized_bias = bias_dataset.map(
    tokenize,
    batched=True
)

Map:   0%|          | 0/29258 [00:00<?, ? examples/s]

In [127]:
tokenized_stance = stance_dataset.map(
    tokenize,
    batched=True
)

Map:   0%|          | 0/48437 [00:00<?, ? examples/s]

In [128]:
print(tokenized_bias[0])

{'text': 'This happens for different reasons , but a key element is the vicious cycle between holding strong attitudes on an issue and something called “ selective perception. ” Essentially , the stronger your views are on an issue like Trump ’ s impeachment , the more likely you are to attend more carefully to information that supports your views and to ignore or disregard information that contradicts them . Consuming more belief-consistent information will , in turn , increase your original support or disapproval for impeachment , which just fortifies your attitudes . So , no , not much change will be seen in the minds of the 33 percent .\nExcept , maybe . One of the more interesting findings from research on attitude change is that our more important , self-defining attitudes do not seem to change incrementally , a little at a time , but they can change dramatically , from one extreme to another . Typically , when others try to change our views on important issues that we hold firml

In [129]:
print(tokenized_stance[0])

{'text': 'Headline: Soldier shot, Parliament locked down after gunfire erupts at war memorial Body: A small meteorite crashed into a wooded area in Nicaragua\'s capital of Managua overnight, the government said Sunday. Residents reported hearing a mysterious boom that left a 16-foot deep crater near the city\'s airport, the Associated Press reports. \r\n\r\nGovernment spokeswoman Rosario Murillo said a committee formed by the government to study the event determined it was a "relatively small" meteorite that "appears to have come off an asteroid that was passing close to Earth." House-sized asteroid 2014 RC, which measured 60 feet in diameter, skimmed the Earth this weekend, ABC News reports. \r\nMurillo said Nicaragua will ask international experts to help local scientists in understanding what happened.\r\n\r\nThe crater left by the meteorite had a radius of 39 feet and a depth of 16 feet,  said Humberto Saballos, a volcanologist with the Nicaraguan Institute of Territorial Studies w

In [130]:
tokenized_bias.save_to_disk(
    "/content/drive/MyDrive/AI-News-Perspective-Analyzer/tokenized_bias"
)

tokenized_stance.save_to_disk(
    "/content/drive/MyDrive/AI-News-Perspective-Analyzer/tokenized_stance"
)

Saving the dataset (0/1 shards):   0%|          | 0/29258 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/48437 [00:00<?, ? examples/s]